# 00 — Inspect data and create the shared datasets
Start here. The existing `TaskData/sequences.json` defines page order. Images are
referenced in portable manifests rather than copied. Split entire triplets once
with seed 42: 64 train / 16 validation; keep the 15 unlabelled test sequences separate.
Run the model notebooks in a kernel with `requirements.txt` installed. Data preparation
and official scoring need only Python; preview cells additionally need Pillow/Matplotlib.

In [ ]:
from pathlib import Path
import os, sys, json
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
elif (ROOT / "manga_notebook_codebase").is_dir():
    ROOT = ROOT / "manga_notebook_codebase"
assert (ROOT / "TaskData/sequences.json").is_file(), "Run from the project or notebooks folder"
sys.path.insert(0, str(ROOT / "src"))
from data_utils import load_jsonl, save_jsonl
from dataset_utils import prepare_datasets, read_dataset, image_paths, save_json
TASK_ROOT = ROOT / "TaskData"
OUTPUTS = ROOT / "outputs"
split = prepare_datasets(ROOT)  # reuses and validates the persisted split
MAX_PIXELS = 256 * 28 * 28  # shared 8 GB GPU budget for inference and training
MAX_NEW_TOKENS = 4096  # inspect raw output if a long sequence reaches the limit
print(ROOT, {k: len(split[k]) for k in ("train", "val")})

In [ ]:
labels = load_jsonl(TASK_ROOT / "development/labels.jsonl")
manifest = json.loads((TASK_ROOT / "sequences.json").read_text(encoding="utf-8"))
print(json.loads((ROOT / "datasets/summary.json").read_text(encoding="utf-8")))
print("Empty labelled pages:", sum(not p for r in labels for p in r["pages"]))
print("Story lines:", sum(len(p) for r in labels for p in r["pages"]))
print("Dataset files:", [str(p.relative_to(ROOT)) for p in (ROOT / "datasets").rglob("*.jsonl")])
assert not set(split["train"]) & set(split["val"])

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
sample = read_dataset(ROOT, "val")[0]
fig, axes = plt.subplots(1, 3, figsize=(18, 10))
for i, (axis, path) in enumerate(zip(axes, image_paths(ROOT, sample))):
    with Image.open(path) as image:
        axis.imshow(image.convert("RGB"))
    axis.set_title(f"Page {i + 1}: {path.name}")
    axis.axis("off")
plt.show()
print(json.dumps(json.loads(sample["target"]), ensure_ascii=False, indent=2))

`datasets/train.jsonl` and `val.jsonl` contain ordered image paths, the common prompt,
and assistant targets. `*_references.jsonl` are strict official-scorer inputs.
`datasets/components/` contains validation-only transcription, shuffled order, and
speaker tasks. `datasets/annotations/` contains unfilled crop and pair annotations.
No test labels are generated. Existing annotations and splits are never overwritten.